In [32]:
# =========================================
#  Detectable Computation via Syndromes
#  (Programs injected between stabilize() rounds)
# =========================================
from __future__ import annotations
import json, csv, time, math, itertools
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np

from qiskit import qasm3
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [33]:
# ---------- USER CONFIG (EDIT THESE) ----------
CONFIG_PATH = Path("../../config.json")   # your IBM Cloud token/instance file
KEY = 'lucia' # can also equal 'lab
OUTDIR = Path("./out_program_leakage") # where artifacts are saved
OUTDIR.mkdir(parents=True, exist_ok=True)

BACKENDS = [
    "ibm_brisbane",
    # add others if you like
]

CODES = ["surface"]       # can add "steane","shor" later
D = 5                     # surface code distance
R = 3                     # rounds of stabilize()
SHOTS = 1000
OPT_LEVEL = 2
LAYOUT_METHODS = ["sabre"]
SEED_TRANSPILE = 20251022  # deterministic transpiler for fair A/B comparisons

# Programs per initial state (we fix logical state for this pivot)
RUNS_PER_PROGRAM = 2
PROGRAM_SET = ["idle", "X", "Z"]   # start with two; you can add more below
STATE_LIST = [{"theta": 0.0, "phi": 0.0}, ]
FIXED_STATE = {"theta": 0.0, "phi": 0.0}  # fixed logical prep


In [ ]:
# ---------- PROGRAM HOOKS (operate on XXZZQubit) ----------
def prog_idle(q):   # do nothing
    return

def prog_logical_X(q):
    # logical X of qtcodes.XXZZQubit
    q.x()

def prog_logical_Z(q):
    q.z()

def prog_logical_Y(q):
    q.z()
    q.x()


PROGRAMS = {
    "idle": prog_idle,
    "X":    prog_logical_X,
    "Z":    prog_logical_Z,
    "Y":    prog_logical_Y,
}

# ---------- HELPERS ----------
def _perm_from_ids(ids):
    """Return a permutation (list of indices) that sorts ids stably."""
    return [i for i, _ in sorted(enumerate(ids), key=lambda t: t[1])]

# ---- canonicalization helpers (if stabilizer IDs available) ----
def _canonicalize_round(bits: str, meta: dict) -> str:
    # Split the round's bitstring into Z- and X-blocks
    if meta["order"] == "ZX":
        z, x = bits[:meta["num_z"]], bits[meta["num_z"]:]
    else:
        x, z = bits[:meta["num_x"]], bits[meta["num_x"]:meta["num_x"] + meta["num_z"]]

    # Build permutations using pure-Python sorting on the IDs
    if "z_ids" in meta and len(meta["z_ids"]) == len(z):
        z_perm = _perm_from_ids(meta["z_ids"])
    else:
        z_perm = list(range(len(z)))

    if "x_ids" in meta and len(meta["x_ids"]) == len(x):
        x_perm = _perm_from_ids(meta["x_ids"])
    else:
        x_perm = list(range(len(x)))

    # Apply permutations
    z_can = "".join(z[i] for i in z_perm)
    x_can = "".join(x[i] for i in x_perm)
    return z_can + x_can


def extract_split_syndromes_canonical(reg2strings: dict[str, list[str]],
                                      syn_reg_names: list[str],
                                      syn_meta: list[dict]) -> tuple[list[str]|None, list[str]|None]:
    """Return (z_joined, x_joined) canonicalized per round. If meta lacks IDs, return (None,None)."""
    have_ids = all(("num_z" in m and "num_x" in m and "order" in m and
                    "z_ids" in m and "x_ids" in m and
                    len(m["z_ids"]) == m["num_z"] and len(m["x_ids"]) == m["num_x"])
                   for m in syn_meta)
    if not have_ids:
        print('NO IDS')
        return None, None

    z_rounds, x_rounds = [], []
    for rn, meta in zip(syn_reg_names, syn_meta):
        raw = reg2strings[rn]  # list[str] for this round
        canon = [_canonicalize_round(s, meta) for s in raw]
        if meta["order"] == "ZX":
            num_z = meta["num_z"]
            z_bits = [s[:num_z] for s in canon]
            x_bits = [s[num_z:]  for s in canon]
        else:
            num_x = meta["num_x"]
            x_bits = [s[:num_x] for s in canon]
            z_bits = [s[num_x:]  for s in canon]
        z_rounds.append(z_bits); x_rounds.append(x_bits)

    shots = len(z_rounds[0]) if z_rounds else 0
    z_joined = ["".join(z_rounds[t][s] for t in range(len(z_rounds))) for s in range(shots)]
    x_joined = ["".join(x_rounds[t][s] for t in range(len(x_rounds))) for s in range(shots)]
    return z_joined, x_joined


In [17]:
# ---------- BUILDERS ----------
def _ids_from_geometry(lattice):
    geo = lattice.geometry  # {'mx': [[anc,n1,n2,n3,n4], ...], 'mz': [...]}

    def _rows_to_ids(rows, tag):
        ids = []
        for row in rows:
            anc = int(row[0])
            nbrs = tuple(-1 if v is None else int(v) for v in row[1:])  # replace None for sortability
            ids.append((tag, anc, *nbrs))  # FLATTENED: ('Z', 3, 10, 11, 15, 16)
        return ids

    x_ids = _rows_to_ids(geo['mx'], 'X')
    z_ids = _rows_to_ids(geo['mz'], 'Z')
    return z_ids, x_ids


def build_surface_circuit(d: int = D,
                          rounds: int = R,
                          program_id: str = "idle",
                          logical_basis: str = "X"
                          ):
    """
    Surface code via qtcodes.XXZZQubit with program hook after each stabilize().

    Encoded initialization:
      - logical_basis="X" -> q.reset_x()   (Hadamard basis, |+_L>)
      - logical_basis="Z" -> q.reset_z()   (Computational basis, |0_L>)

    Optionally, set use_physical_u=True to also apply a *single-physical-qubit*
    U(theta,phi,lambda) on qubit 0 BEFORE stabilizing (baseline/unencoded experiments).
    """
    from qtcodes import XXZZQubit

    basis_tag = logical_basis.upper()
    assert basis_tag in ("X", "Z"), "logical_basis must be 'X' or 'Z'"

    name = f"surface_d{d}_basis{basis_tag}_{program_id}"
    q = XXZZQubit({'d': d}, name=name)

    # --- ENCODED INITIALIZATION ---
    if basis_tag == "X":
        q.reset_x()   # logical |+_L>
    else:
        q.reset_z()   # logical |0_L>

    syn_reg_names, syn_meta = [], []
    program_fn = PROGRAMS[program_id]

    for _ in range(rounds):
        # 1) Stabilize (measure syndromes)
        q.stabilize()

        # 2) Record meta for canonicalization (now using geometry-based IDs)
        T = q.lattice.params["T"]
        creg = q.lattice.cregisters[f"syndrome{T}"]
        num_syn = q.lattice.params["num_syn"]
        num_z = int(num_syn[q.lattice.SYNZ]); num_x = int(num_syn[q.lattice.SYNX])

        z_ids, x_ids = _ids_from_geometry(q.lattice)    # <<-- NEW: stable IDs from geometry
        assert len(z_ids) == num_z and len(x_ids) == num_x, "ID count mismatch vs num_syn"

        syn_reg_names.append(creg.name)
        syn_meta.append({
            "order": "ZX",
            "num_z": num_z, "num_x": num_x,
            "z_ids": z_ids, "x_ids": x_ids,
        })

        # 3) Computation/program between rounds
        program_fn(q)

    return q.circ, syn_reg_names, syn_meta

CODE_BUILDERS = {
    "surface": build_surface_circuit,
    # "steane": build_steane_circuit,  # add later
    # "shor":   build_shor_circuit,    # add later
}

In [ ]:
# ---------- PARAM GRID (programs × runs) ----------
LOGICAL_BASES = ["X", "Z"]        # or ["X", "Z"]
PROGRAM_SET = list(PROGRAMS)        # e.g., ["idle","X","Z","Y"]

def make_params_list() -> list[dict]:
    params = []
    for basis in LOGICAL_BASES:
        for prog in PROGRAM_SET:
            for r in range(RUNS_PER_PROGRAM):
                params.append({
                    "logical_basis": basis,
                    "program_id": prog,
                    "run_id": r,
                })
    return params

In [30]:
def build_and_pack_circuits(code_name: str, params_list: list[dict]):
    builder = CODE_BUILDERS[code_name]
    metadata, circuit_objs = [], []
    for pp in params_list:
        run_id       = pp.get("run_id", 0)
        program_id   = pp.get("program_id", "idle")
        logical_basis= pp.get("logical_basis", "X")

        qc, syn_regs, syn_meta = builder(
            d=D, rounds=R,
            program_id=program_id,
            logical_basis=logical_basis,
        )

        metadata.append({
            "code": code_name,
            "program_id": program_id,
            "run_id": run_id,
            "logical_basis": logical_basis,
            "syn_reg_names": syn_regs,
            "syn_meta": syn_meta,
            "qasm3": qasm3.dumps(qc),
        })
        circuit_objs.append(qc)
    return metadata, circuit_objs


In [ ]:
def build_tag(backend_name: str, code: str, basis: str, program: str) -> str:
    return f"{backend_name}_{code}_basis{basis}_{program}"

def main():
    import pickle as p
    # Auth
    with open(CONFIG_PATH) as f:
        cfg = json.load(f)[KEY]
    service = QiskitRuntimeService(
        channel="ibm_cloud",
        token=cfg["api_key"],
        instance=cfg["ibm_instance_crn"],
    )

    params = make_params_list()

    for backend_name in BACKENDS:
        backend = service.backend(backend_name)
        print(f"\n== Backend: {backend_name} ==")

        # ---------- build all circuits ----------
        all_metadata, all_circuits = [], []
        for code in CODES:
            md, circs = build_and_pack_circuits(code, params)
            all_metadata.extend(md); all_circuits.extend(circs)

        # ---------- transpile (deterministic) ----------
        transpiled, saved_md = [], []
        rng = np.random.default_rng(12345)

        for meta, qc in zip(all_metadata, all_circuits):
            code   = meta["code"]
            prog   = meta["program_id"]
            basis  = meta["logical_basis"]
            run_id = meta["run_id"]

            base_tag = build_tag(backend_name, code, basis, prog)
            pkl_path = OUTDIR / f"{base_tag}_transpiled.pkl"

            if not pkl_path.exists() and run_id == 0:
                # --- first run: transpile and save ---
                initial_layout = list(rng.permutation(qc.num_qubits))
                pm = generate_preset_pass_manager(
                    optimization_level=OPT_LEVEL,
                    backend=backend,
                    layout_method=LAYOUT_METHODS[0],
                    initial_layout=initial_layout,
                    seed_transpiler=SEED_TRANSPILE,
                )
                tqc = pm.run(qc)
                with open(pkl_path, "wb") as f:
                    p.dump({"tqc": tqc, "initial_layout": initial_layout}, f)
                print(f"Saved transpiled circuit for {prog} (basis={basis}) -> {pkl_path.name}")

            else:
                with open(pkl_path, "rb") as f:
                    data = p.load(f)
                tqc = data["tqc"]
                initial_layout = data["initial_layout"]
                print(f"Loaded cached circuit for {prog} (basis={basis}) <- {pkl_path.name}")

            transpiled.append(tqc)
            m = dict(meta)
            m["backend"] = backend_name
            m["layout_method"] = LAYOUT_METHODS[0]
            m["initial_layout"] = initial_layout
            m["logical_basis"] = basis
            saved_md.append(m)

        # Save circuits bundle
        pkl_path = OUTDIR / f"{backend_name}_circuits_programs.pkl"
        with open(pkl_path, "wb") as f:
            import pickle as p
            p.dump((saved_md, transpiled), f)
        print(f"==\tSaved circuit bundle to: {pkl_path} ==")

        # ---------- run ----------
        sampler = Sampler(mode=backend)
        print(f"\tSubmitting {len(transpiled)} circuits to {backend_name} with {SHOTS} shots each …")
        job = sampler.run(transpiled, shots=SHOTS)
        submit_ts = time.time()
        results = job.result()
        print("\t\tJob complete:", job.job_id())

        # ---------- aggregate per (code, program, run) ----------
        per_prog_counts_all_str = defaultdict(Counter)  # canonicalized strings (Z+X)
        per_shot_rows = []  # optional detailed CSV

        for idx, res in enumerate(results):
            meta = saved_md[idx]
            key = (meta["code"], meta["program_id"], meta["run_id"], meta["logical_basis"])
            syn_regs = meta["syn_reg_names"]
            syn_meta = meta["syn_meta"]

            # map reg->list[str]
            reg2strings = {reg: vals.get_bitstrings() for reg, vals in res.data.items()}

            # canonical strings (if IDs present)
            z_joined, x_joined = extract_split_syndromes_canonical(reg2strings, syn_regs, syn_meta)
            if z_joined is not None:
                combined = [z + x for z, x in zip(z_joined, x_joined)]
                per_prog_counts_all_str[key].update(combined)

                # optional per-shot logging
                for sZ, sX in zip(z_joined, x_joined):
                    per_shot_rows.append({
                        "backend": meta["backend"], "code": meta["code"],
                        "program": meta["program_id"], "run": meta["run_id"],
                        "logical_basis": meta["logical_basis"],
                        "syndrome_Z": sZ, "syndrome_X": sX
                    })

        # ---------- save raw counts ----------
        raw_json = {
            "backend": backend_name,
            "shots": SHOTS,
            "rounds": R,
            "timestamp": submit_ts,
            "counts_all_str": {
                f"{c}|prog={p}|run={r}|basis={b}": dict(cnt)
                for (c, p, r, b), cnt in per_prog_counts_all_str.items()
            },
        }

        with open(OUTDIR / f"{backend_name}_syndrome_counts_program.json", "w") as f:
            json.dump(raw_json, f, indent=2)

        if per_shot_rows:
            with open(OUTDIR / f"{backend_name}_per_shot_program.csv", "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=["backend","code","program","run","logical_basis","syndrome_Z","syndrome_X"])
                w.writeheader(); w.writerows(per_shot_rows)

        print(f"Saved: {pkl_path.name}, {backend_name}_syndrome_counts_program.json")

In [ ]:
if __name__ == "__main__":
    main()


== Backend: ibm_brisbane ==
Saved transpiled circuit for idle to ibm_brisbane_surface_theta0.000_phi0.000_idle_transpiled.pkl
Loaded cached transpiled circuit for idle from ibm_brisbane_surface_theta0.000_phi0.000_idle_transpiled.pkl
Saved transpiled circuit for X to ibm_brisbane_surface_theta0.000_phi0.000_X_transpiled.pkl
Loaded cached transpiled circuit for X from ibm_brisbane_surface_theta0.000_phi0.000_X_transpiled.pkl
Saved transpiled circuit for Z to ibm_brisbane_surface_theta0.000_phi0.000_Z_transpiled.pkl
Loaded cached transpiled circuit for Z from ibm_brisbane_surface_theta0.000_phi0.000_Z_transpiled.pkl
==	Saved ciccuit to: out_program_leakage/ibm_brisbane_circuits_programs.pkl ==
	Submitting 6 circuits to ibm_brisbane with 1000 shots each …
		Job complete: d3t6db1sg33c73de286g
Appended 15 rows -> tv_program_merged.csv
Saved: ibm_brisbane_circuits_programs.pkl, ibm_brisbane_syndrome_counts_program.json, per_shot_program.csv, tv_program_merged.csv
